In [1]:
import re
import sys
sys.path.insert(0, "..")
from src.preprocessing.segmenter import segment_document
import fitz

# Charger le CV d'Esra
doc = fitz.open("../data/raw/cvs/257264722 CV.pdf")
text_cv = "".join(page.get_text() for page in doc)
seg_cv = segment_document(text_cv)
education_text = seg_cv.sections.get("EDUCATION", "")
print("Section EDUCATION extraite :")
print(education_text)
print(f"\nLongueur : {len(education_text)} chars")

Section EDUCATION extraite :
Master of Science in Cybersecurity - Summer 2024
New York University, New York

Bachelor of Science in Software Engineering - Spring 2017
AlHosn University, Abu Dhabi, UAE

Longueur : 171 chars


In [2]:
GPA_PATTERNS = [
    # Explicites avec échelle — du plus spécifique au plus général
    (r"(?:GPA|CGPA|Grade Point Average)[:\s]+(\d+\.?\d*)\s*/\s*(\d+\.?\d*)", "explicit_scale"),
    (r"(\d+\.?\d*)\s*/\s*(4\.0|4)", "scale_4"),
    (r"(\d+\.?\d*)\s*/\s*(5\.0|5)", "scale_5"),
    (r"(\d+\.?\d*)\s*/\s*(20)", "scale_20"),
    (r"(\d+\.?\d*)\s*/\s*(100)", "scale_100"),
    # Implicites — échelle 4.0 supposée
    (r"(?:GPA|CGPA)[:\s]+(\d+\.?\d*)", "implicit_4"),
    (r"(?:cumulative|overall)\s+(?:GPA|average)[:\s]+(\d+\.?\d*)", "implicit_4"),
    # Mentions d'honneur
    (r"(Summa Cum Laude)", "honour"),
    (r"(Magna Cum Laude)", "honour"),
    (r"(Cum Laude)", "honour"),
]

HONOUR_MAP = {
    "summa cum laude": 3.9,
    "magna cum laude": 3.7,
    "cum laude":       3.5,
}

def normalise_gpa(raw: float, scale: float) -> float:
    """Normalise un GPA vers l'échelle 4.0."""
    if scale == 4.0:
        return round(raw, 3)
    elif scale == 5.0:
        return round(raw * 0.8, 3)
    elif scale == 20.0:
        return round(raw * 0.2, 3)
    elif scale == 100.0:
        return round(raw * 0.04, 3)
    else:
        return round(raw / scale * 4.0, 3)

print("Fonctions GPA définies.")

Fonctions GPA définies.


In [3]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class GPAResult:
    raw_value: Optional[float]
    original_scale: Optional[float]
    normalised_gpa: Optional[float]
    confidence: float
    source: str          # explicit_scale | implicit_4 | honour | not_found
    raw_match: Optional[str]
    degree_level: str    # bsc | msc | unknown

def extract_gpa(text: str, degree_level: str = "unknown") -> GPAResult:
    """
    Extrait le GPA d'un texte de section EDUCATION.
    Retourne un GPAResult avec valeur brute, échelle, normalisée et confiance.
    """
    if not text:
        return GPAResult(None, None, None, 0.0, "not_found", None, degree_level)

    for pattern, pattern_type in GPA_PATTERNS:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            if pattern_type == "honour":
                honour_text = match.group(1).lower()
                normalised = HONOUR_MAP.get(honour_text, 3.5)
                return GPAResult(
                    raw_value=normalised,
                    original_scale=4.0,
                    normalised_gpa=normalised,
                    confidence=0.60,
                    source="honour",
                    raw_match=match.group(0),
                    degree_level=degree_level
                )
            elif pattern_type == "explicit_scale":
                raw = float(match.group(1))
                scale = float(match.group(2))
                return GPAResult(
                    raw_value=raw,
                    original_scale=scale,
                    normalised_gpa=normalise_gpa(raw, scale),
                    confidence=0.95,
                    source="explicit_scale",
                    raw_match=match.group(0),
                    degree_level=degree_level
                )
            elif pattern_type in ("scale_4", "scale_5", "scale_20", "scale_100"):
                raw = float(match.group(1))
                scale = float(match.group(2))
                return GPAResult(
                    raw_value=raw,
                    original_scale=scale,
                    normalised_gpa=normalise_gpa(raw, scale),
                    confidence=0.90,
                    source="explicit_scale",
                    raw_match=match.group(0),
                    degree_level=degree_level
                )
            elif pattern_type == "implicit_4":
                raw = float(match.group(1))
                return GPAResult(
                    raw_value=raw,
                    original_scale=4.0,
                    normalised_gpa=normalise_gpa(raw, 4.0),
                    confidence=0.75,
                    source="implicit_4",
                    raw_match=match.group(0),
                    degree_level=degree_level
                )

    return GPAResult(None, None, None, 0.0, "not_found", None, degree_level)

print("Extracteur GPA défini.")

Extracteur GPA défini.


In [4]:
# Test 1 — CV Esra (GPA absent)
print("=== CV ESRA AL-NASHASH ===")
result_esra = extract_gpa(education_text, degree_level="bsc")
print(f"GPA trouvé    : {result_esra.raw_value}")
print(f"Confiance     : {result_esra.confidence}")
print(f"Source        : {result_esra.source}")
print(f"→ Attendu     : not_found (GPA absent du CV)")

# Test 2 — Transcript Devon (GPA 3.80/4.0)
print("\n=== TRANSCRIPT DEVON SERRAO ===")
transcript_text = """
TOTAL INSTITUTION  30.00  30.00  114.0  3.80
OVERALL            30.00  30.00  114.0  3.80
GPA: 3.80
"""
result_devon = extract_gpa(transcript_text, degree_level="msc")
print(f"GPA trouvé    : {result_devon.raw_value}")
print(f"Échelle       : {result_devon.original_scale}")
print(f"Normalisé     : {result_devon.normalised_gpa}")
print(f"Confiance     : {result_devon.confidence}")
print(f"→ Attendu     : 3.80 / 4.0")

# Test 3 — GPA sur échelle 20 (cas test)
print("\n=== TEST ÉCHELLE /20 ===")
test_text_20 = "Moyenne générale : 16.4 / 20"
result_20 = extract_gpa(test_text_20)
print(f"GPA brut      : {result_20.raw_value}")
print(f"Échelle       : {result_20.original_scale}")
print(f"Normalisé /4  : {result_20.normalised_gpa}")
print(f"→ Attendu     : 3.28 / 4.0")

# Test 4 — Mention d'honneur
print("\n=== TEST MENTION D'HONNEUR ===")
test_honour = "Graduated Summa Cum Laude, Spring 2020"
result_honour = extract_gpa(test_honour)
print(f"GPA estimé    : {result_honour.normalised_gpa}")
print(f"Confiance     : {result_honour.confidence}")
print(f"→ Attendu     : 3.9 / confiance 0.60")


=== CV ESRA AL-NASHASH ===
GPA trouvé    : None
Confiance     : 0.0
Source        : not_found
→ Attendu     : not_found (GPA absent du CV)

=== TRANSCRIPT DEVON SERRAO ===
GPA trouvé    : 3.8
Échelle       : 4.0
Normalisé     : 3.8
Confiance     : 0.75
→ Attendu     : 3.80 / 4.0

=== TEST ÉCHELLE /20 ===
GPA brut      : 16.4
Échelle       : 20.0
Normalisé /4  : 3.28
→ Attendu     : 3.28 / 4.0

=== TEST MENTION D'HONNEUR ===
GPA estimé    : 3.9
Confiance     : 0.6
→ Attendu     : 3.9 / confiance 0.60


In [1]:
import pandas as pd
df = pd.read_csv("../data/qs_rankings_2025.csv")
print(f"Universités chargées : {len(df)}")
print(df[df['university_name'].str.contains('York|Chester|Missouri', case=False)])

Universités chargées : 62
    rank                      university_name         country
21    22             University of Manchester  United Kingdom
31    32                  New York University   United States
60   601                University of Chester  United Kingdom
61   801  Southeast Missouri State University   United States
